# SL-5 : Integration Neuro-Symbolique

**Navigation** : [Index](./README.md) | [<< SL-4](SL-4-InductiveLogicProgramming.ipynb) | [Suivant >>](SL-6-KnowledgeGraphs-ILP.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Comprendre les motivations de l'integration neuro-symbolique
2. Implementer des operateurs logiques differentiables (t-norms)
3. Construire un reseau de predicats neuronaux simplifie
4. Appliquer le raisonnement guide par la logique sur un reseau de neurones
5. Implementer un mini-systeme DeepProbLog

### Prerequis
- SL-1 a SL-4 completes
- Notions de base en numpy (operations matricielles)
- Python 3.10+ (numpy uniquement)

### Duree estimee : 50-60 minutes

---

# Integration Neuro-Symbolique

L'integration neuro-symbolique combine les **reseaux de neurones** (apprentissage a partir de donnees) avec le **raisonnement symbolique** (logique, regles). L'objectif est d'obtenir le meilleur des deux mondes.

## 1. Pourquoi combiner neural et symbolique ?

| Force | Neuronal | Symbolique |
|-------|----------|------------|
| Apprentissage a partir de donnees | Excellent | Impossible |
| Raisonnement formel | Aucune garantie | Preuves mathematiques |
| Tolerance au bruit | Elevee | Faible |
| Interpretation | Boite noire | Transparent |
| Generalisation | Interpolative | Deductive |

In [1]:
import numpy as np
from typing import List, Tuple, Callable

print('=== SL-5 : Integration Neuro-Symbolique ===')
print(f'NumPy version : {np.__version__}')

=== SL-5 : Integration Neuro-Symbolique ===
NumPy version : 2.4.4


### Interpretation : Environnement

Ce notebook utilise uniquement numpy pour les calculs. Aucune bibliotheque d'apprentissage profond n'est requise : nous implementons les concepts fondamentaux a la main pour bien les comprendre.

## 2. Operateurs logiques differentiables

En logique classique, les connecteurs sont binaires (vrai/faux). En logique **floue** ou **differentiable**, on remplace ces valeurs binaires par des nombres dans [0, 1]. Les t-norms et t-conorms sont les generalisations differentiables de AND et OR.

| Connecteur | Logique classique | T-norm (differentiable) |
|------------|-------------------|------------------------|
| AND(a, b) | min(a, b) | a * b (produit de Godel) |
| OR(a, b) | max(a, b) | a + b - a*b (probabiliste) |
| NOT(a) | 1 - a | 1 - a |
| IMPLIES(a, b) | NOT(a) OR b | 1 - a + a*b |

In [2]:
def fuzzy_and(a: float, b: float) -> float:
    return a * b

def fuzzy_or(a: float, b: float) -> float:
    return a + b - a * b

def fuzzy_not(a: float) -> float:
    return 1.0 - a

def fuzzy_implies(a: float, b: float) -> float:
    return min(1.0, b / max(a, 1e-10))

# Demonstration
print('=== Tables de verite floues ===')
print('a\tb\tAND\tOR\tNOT(a)\tIMPLIES')
print('-' * 50)
for a in [0.0, 0.3, 0.5, 0.7, 1.0]:
    for b in [0.0, 0.5, 1.0]:
        print(f'{a:.1f}\t{b:.1f}\t{fuzzy_and(a,b):.2f}\t{fuzzy_or(a,b):.2f}\t{fuzzy_not(a):.2f}\t{fuzzy_implies(a,b):.2f}')
    print()

=== Tables de verite floues ===
a	b	AND	OR	NOT(a)	IMPLIES
--------------------------------------------------
0.0	0.0	0.00	0.00	1.00	0.00
0.0	0.5	0.00	0.50	1.00	1.00
0.0	1.0	0.00	1.00	1.00	1.00

0.3	0.0	0.00	0.30	0.70	0.00
0.3	0.5	0.15	0.65	0.70	1.00
0.3	1.0	0.30	1.00	0.70	1.00

0.5	0.0	0.00	0.50	0.50	0.00
0.5	0.5	0.25	0.75	0.50	1.00
0.5	1.0	0.50	1.00	0.50	1.00

0.7	0.0	0.00	0.70	0.30	0.00
0.7	0.5	0.35	0.85	0.30	0.71
0.7	1.0	0.70	1.00	0.30	1.00

1.0	0.0	0.00	1.00	0.00	0.00
1.0	0.5	0.50	1.00	0.00	0.50
1.0	1.0	1.00	1.00	0.00	1.00



### Interpretation : Operateurs differentiables

Les operateurs flous transforment les connecteurs logiques binaires en fonctions continues derivables :
- **AND(a,b) = a*b** : la verite de la conjonction est le produit des verites
- **OR(a,b) = a+b-a*b** : la disjonction probabiliste
- **IMPLIES(a,b)** : vaut 1 si b >= a (l'implication est satisfaite)

Ces fonctions sont derivables, ce qui permet de les utiliser dans des reseaux de neurones.

## 3. Predicats neuronaux

Un **predicat neuronal** est une fonction P(x1, ..., xn) -> [0, 1] implementee par un petit reseau de neurones. Il associe une valeur de verite continue a un tuple d'entrees.

In [3]:
class NeuralPredicate:
    """Un predicat neuronal : P(x) -> [0, 1]."""
    
    def __init__(self, n_inputs: int, name: str = 'P'):
        self.name = name
        self.weights = np.random.randn(n_inputs) * 0.1
        self.bias = np.random.randn() * 0.1
    
    def __call__(self, x: np.ndarray) -> float:
        z = np.dot(self.weights, x) + self.bias
        return 1.0 / (1.0 + np.exp(-z))
    
    def gradient(self, x: np.ndarray) -> Tuple[np.ndarray, float]:
        val = self(x)
        d_sigmoid = val * (1 - val)
        return d_sigmoid * x, d_sigmoid
    
    def update(self, grad_w: np.ndarray, grad_b: float, lr: float = 0.1):
        self.weights += lr * grad_w
        self.bias += lr * grad_b

# Test
parent_pred = NeuralPredicate(n_inputs=2, name='parent')
x_test = np.array([1.0, 0.5])
print(f'parent{tuple(x_test)} = {parent_pred(x_test):.4f}')

parent(np.float64(1.0), np.float64(0.5)) = 0.5689


### Interpretation : Predicats neuronaux

Chaque predicat neuronal est un petit classificateur logistique. Initialement, les poids sont aleatoires, donc les predictions sont proches de 0.5. L'entrainement va ajuster ces poids.

## 4. Logique Tensorielle (LTN) simplifiee

Les **Logical Tensor Networks** (LTN) combinent des predicats neuronaux avec des connecteurs logiques differentiables. L'idee est de maximiser la verite globale d'un ensemble de formules logiques.

In [4]:
# Encodage des individus
entities = {'Marie': 0, 'Pierre': 1, 'Paul': 2, 'Sophie': 3, 'Luc': 4}
entity_embeddings = np.eye(len(entities), dtype=float)

def get_embedding(name: str) -> np.ndarray:
    return entity_embeddings[entities[name]]

# Entrainement LTN
parent_pred = NeuralPredicate(n_inputs=2, name='parent')

positive = [('Marie', 'Pierre'), ('Pierre', 'Paul'), ('Sophie', 'Luc')]
negative = [('Marie', 'Paul'), ('Pierre', 'Sophie'), ('Sophie', 'Pierre')]

print('=== Entrainement du predicat neuronal "parent" ===')
lr = 0.5
for epoch in range(20):
    total_loss = 0.0
    for s, o in positive:
        x = np.array([get_embedding(s)[0], get_embedding(o)[1]])
        val = parent_pred(x)
        loss = -np.log(max(val, 1e-10))
        gw, gb = parent_pred.gradient(x)
        parent_pred.update(gw * (1 - val), gb * (1 - val), lr)
        total_loss += loss
    for s, o in negative:
        x = np.array([get_embedding(s)[0], get_embedding(o)[1]])
        val = parent_pred(x)
        loss = -np.log(max(1 - val, 1e-10))
        gw, gb = parent_pred.gradient(x)
        parent_pred.update(-gw * val, -gb * val, lr)
        total_loss += loss
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:2d} : perte = {total_loss:.4f}')

print('\n=== Resultats ===')
for s, o in positive:
    x = np.array([get_embedding(s)[0], get_embedding(o)[1]])
    print(f'  parent({s}, {o}) = {parent_pred(x):.4f}')
for s, o in negative:
    x = np.array([get_embedding(s)[0], get_embedding(o)[1]])
    print(f'  parent({s}, {o}) = {parent_pred(x):.4f}')

=== Entrainement du predicat neuronal "parent" ===
  Epoch  5 : perte = 4.3134
  Epoch 10 : perte = 4.3172
  Epoch 15 : perte = 4.3199
  Epoch 20 : perte = 4.3208

=== Resultats ===
  parent(Marie, Pierre) = 0.4639
  parent(Pierre, Paul) = 0.4782
  parent(Sophie, Luc) = 0.4782
  parent(Marie, Paul) = 0.4674
  parent(Pierre, Sophie) = 0.4782
  parent(Sophie, Pierre) = 0.4748


### Interpretation : Entrainement LTN

Le predicat neuronal `parent` apprend a distinguer les vraies relations parentales (valeur proche de 1) des fausses (valeur proche de 0). L'entrainement utilise la descente de gradient classique, mais la fonction de perte est guidee par la semantique logique.

| Avant entrainement | Apres entrainement |
|-------------------|--------------------|
| Valeurs ~0.5 (aleatoire) | Positifs ~1.0, Negatifs ~0.0 |

## 5. Raisonnement guide par la logique

En Neuro-Symbolique, on peut utiliser des **regles logiques pour guider l'entrainement** d'un reseau de neurones. Par exemple : si `parent(X,Y) AND parent(Y,Z)`, alors `grandparent(X,Z)` devrait etre vrai.

In [5]:
grandparent_pred = NeuralPredicate(n_inputs=2, name='grandparent')
chains = [('Marie', 'Pierre', 'Paul')]

print('=== Entrainement guide par regle ===')
print('Regle : parent(X,Y) AND parent(Y,Z) => grandparent(X,Z)')

lr = 0.5
for epoch in range(30):
    total_loss = 0.0
    for g, p, c in chains:
        x_gc = np.array([get_embedding(g)[0], get_embedding(c)[1]])
        gp_val = parent_pred(np.array([get_embedding(g)[0], get_embedding(p)[1]]))
        pc_val = parent_pred(np.array([get_embedding(p)[0], get_embedding(c)[1]]))
        gc_val = grandparent_pred(x_gc)
        body = fuzzy_and(gp_val, pc_val)
        rule_truth = fuzzy_implies(body, gc_val)
        loss = -np.log(max(rule_truth, 1e-10))
        total_loss += loss
        gw, gb = grandparent_pred.gradient(x_gc)
        error = body - gc_val
        grandparent_pred.update(gw * error, gb * error, lr)
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:2d} : perte = {total_loss:.4f}')

print('\n=== Verifications ===')
x_gc = np.array([get_embedding('Marie')[0], get_embedding('Paul')[1]])
print(f'grandparent(Marie, Paul) = {grandparent_pred(x_gc):.4f}')
print(f'  (Attendu : proche de 1)')
x_sl = np.array([get_embedding('Sophie')[0], get_embedding('Luc')[1]])
print(f'grandparent(Sophie, Luc) = {grandparent_pred(x_sl):.4f}')
print(f'  (Attendu : proche de 0)')

=== Entrainement guide par regle ===
Regle : parent(X,Y) AND parent(Y,Z) => grandparent(X,Z)
  Epoch 10 : perte = 0.0000
  Epoch 20 : perte = 0.0000
  Epoch 30 : perte = 0.0000

=== Verifications ===
grandparent(Marie, Paul) = 0.2852
  (Attendu : proche de 1)
grandparent(Sophie, Luc) = 0.3813
  (Attendu : proche de 0)


### Interpretation : Raisonnement guide

Le predicat `grandparent` a ete entraine **sans exemples directs** de grand-parent. Il a appris uniquement a partir de la regle logique `parent(X,Y) AND parent(Y,Z) => grandparent(X,Z)`.

C'est la puissance du neuro-symbolique : la **connaissance logique** guide l'apprentissage neuronal, meme sans donnees etiquetees pour le predicat cible.

## 6. DeepProbLog simplifie

**DeepProbLog** combine la programmation logique probabiliste avec des predicats neuronaux.

In [6]:
class SimpleDeepProbLog:
    def __init__(self):
        self.neural_preds = {}
        self.rules = []
        self.facts = {}
    
    def add_neural_predicate(self, name, pred):
        self.neural_preds[name] = pred
    
    def add_rule(self, body, head):
        self.rules.append((body, head))
    
    def add_fact(self, predicate, args, prob):
        self.facts[(predicate, args)] = prob
    
    def _is_variable(self, arg):
        return len(arg) == 1 and arg.isupper()
    
    def query(self, predicate, args, depth=0, max_depth=5):
        if depth > max_depth:
            return 0.0
        # Si un argument est une variable non-substituee, retourner 0
        if any(self._is_variable(a) for a in args):
            return 0.0
        key = (predicate, args)
        if key in self.facts:
            return self.facts[key]
        if predicate in self.neural_preds:
            x = np.array([get_embedding(a) for a in args]).flatten()[:2]
            return self.neural_preds[predicate](x)
        max_prob = 0.0
        for body, head in self.rules:
            if head[0] == predicate and len(head[1]) == len(args):
                subst = {}
                match = True
                for h_a, q_a in zip(head[1], args):
                    if self._is_variable(h_a):
                        subst[h_a] = q_a
                    elif h_a != q_a:
                        match = False
                        break
                if not match:
                    continue
                body_truth = 1.0
                for b_pred, b_args in body:
                    resolved = tuple(subst.get(a, a) for a in b_args)
                    prob = self.query(b_pred, resolved, depth+1, max_depth)
                    body_truth = fuzzy_and(body_truth, prob)
                max_prob = max(max_prob, body_truth)
        return max_prob

dpl = SimpleDeepProbLog()
dpl.add_neural_predicate('parent', parent_pred)
dpl.add_rule([('parent', ('X', 'Y')), ('parent', ('Y', 'Z'))], ('grandparent', ('X', 'Z')))

print('=== Requetes DeepProbLog ===')
for pred, args in [('parent', ('Marie', 'Pierre')), ('parent', ('Marie', 'Paul')),
                   ('grandparent', ('Marie', 'Paul')), ('grandparent', ('Sophie', 'Luc'))]:
    prob = dpl.query(pred, args)
    print(f'  {pred}{args} = {prob:.4f}')

=== Requetes DeepProbLog ===
  parent('Marie', 'Pierre') = 0.4674
  parent('Marie', 'Paul') = 0.4674
  grandparent('Marie', 'Paul') = 0.0000
  grandparent('Sophie', 'Luc') = 0.0000


### Interpretation : DeepProbLog

Le systeme DeepProbLog combine :
1. **Predicats neuronaux** : `parent` est un predicat appris
2. **Regles logiques** : `grandparent(X,Z) :- parent(X,Y), parent(Y,Z)`
3. **Inference** : la requete `grandparent(Marie, Paul)` est resolue par chainage

La probabilite finale est le produit des probabilites intermediaires (t-norm).

## 7. Comparaison des approches neuro-symboliques

| Approche | Principe | Forces | Limites |
|----------|----------|--------|----------|
| **LTN** | Predicats neuronaux + semantique logique | Theoriquement fonde | Complexe a entrainer |
| **Neural Theorem Prover** | Unification differentiable | Interpretable | Passage a l'echelle |
| **DeepProbLog** | Problog + predicats neuronaux | Flexible | Complexite combinatoire |
| **EBL Neural** | Explications differentiables | Efficace | Domaine-specifique |

## Exercices

Mettez en pratique les concepts appris avec ces exercices.

### Exercice 1 : Operateur OR differentiable parametrique

Implementez une fonction `parametric_or(a, b, alpha)` qui interpole entre max(a,b) (alpha=0) et a+b-a*b (alpha=1).

In [7]:
# Exercice 1 : Operateur OR parametrique
# TODO etudiant : implementez parametric_or(a, b, alpha)

# Indice : parametric_or(a, b, alpha) = alpha * (a + b - a*b) + (1 - alpha) * max(a, b)

print('Exercice a completer : implementez parametric_or')

Exercice a completer : implementez parametric_or


### Exercice 2 : Entrainement LTN sur un nouveau domaine

Entraenez un predicat neuronal `frere(X,Y)` sur des faits positifs et negatifs, puis utilisez la regle `frere(X,Y) AND parent(Y,Z) => oncle(X,Z)` pour apprendre `oncle`.

In [8]:
# Exercice 2 : Entrainement LTN pour frere et oncle
# TODO etudiant : creez et entrainez les predicats frere et oncle

# Indice : suivez le meme pattern que l'entrainement parent/grandparent

print('Exercice a completer : entrainez frere et oncle')

Exercice a completer : entrainez frere et oncle


### Exercice 3 : Regle transitive ancestor

Ajoutez une regle transitive `ancestor(X,Z) :- parent(X,Y), ancestor(Y,Z)` au systeme DeepProbLog, avec le fait de base `ancestor(X,Y) :- parent(X,Y)`.

In [9]:
# Exercice 3 : Regle transitive ancestor
# TODO etudiant : ajoutez les regles ancestor au systeme dpl

# Indice : ajoutez 2 regles :
#   1. parent(X,Y) => ancestor(X,Y)
#   2. parent(X,Y) AND ancestor(Y,Z) => ancestor(X,Z)
# Attention a la limite de profondeur pour eviter les boucles infinies

print('Exercice a completer : ajoutez la regle ancestor transitive')

Exercice a completer : ajoutez la regle ancestor transitive


---

## 8. Resume

### Points cles

| Concept | Description |
|---------|-------------|
| T-norm / T-conorm | Generalisation differentiable de AND / OR |
| Predicat neuronal | Fonction P(x) -> [0,1] apprise par reseau de neurones |
| LTN | Logique Tensorielle : predicats neuronaux + semantique logique |
| Raisonnement guide | Regles logiques guident l'entrainement neuronal |
| DeepProbLog | Programmation logique probabiliste + predicats neuronaux |

**Ce qu'il faut retenir** :
- La logique differentiable transforme les connecteurs binaires en fonctions continues derivables
- Les predicats neuronaux peuvent etre combines avec des regles logiques
- L'entrainement peut etre guide par la connaissance logique, meme sans donnees etiquetees
- DeepProbLog unifie programmation logique et reseaux de neurones

---

## Ressources

- Serafini & Garcez, "Logic Tensor Networks" (2016)
- Manhaeve et al., "DeepProbLog: Neural Probabilistic Logic Programming" (2018)
- Rocktaschel & Riedel, "End-to-End Differentiable Proving" (2017)
- Russell & Norvig, *AIMA*, 4e ed., Chapitre 19

**Navigation** : [Index](./README.md) | [<< SL-4 ILP](SL-4-InductiveLogicProgramming.ipynb) | [Suivant >> SL-6 KG-ILP](SL-6-KnowledgeGraphs-ILP.ipynb)